# Layer-wise Adversarial Machine Learning Detector

## Configuration

In [16]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn


import libs.preprocess as pp
import libs.lwad_wrapper as lw
import libs.lwad_attack as la
import libs.lwad_trainer as lt
import libs.lwad_evaluator as le

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Config
DATASET_PATH = "../dataset/unsw-nb15/"
DOWNLOAD_DATASET = False
VERBOSE = True

from libs.lwad_config import EPOCHS, BATCH_SIZE, \
    LR, LR_DET, LAMBDA_DET, TASK_LOSS_ON_ADV, \
    THRESHOLD, SEED, CHECKPOINT, PATIENCE, MIN_DELTA

In [18]:
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


## Preprocessing

In [19]:
X_train, y_train, X_val, y_val, X_test, y_test, feature_names, attack_mask = pp.load_unsw(DATASET_PATH, DOWNLOAD_DATASET, True)

X_train, y_train = X_train.to(device), y_train.to(device)
X_val, y_val = X_val.to(device), y_val.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)
attack_mask = attack_mask.to(device)

Path to dataset files: ../dataset/unsw-nb15

Training set size: (57632, 44)
Validation set size: (24700, 44)
Test set size: (175341, 44)

Statification on attack categories:

            Train  Validation   Test
attack_cat                          
6.0         25900       11100  56000
5.0         13210        5661  40000
3.0          7792        3340  33393
4.0          4243        1819  18184
2.0          2862        1227  12264
7.0          2447        1049  10491
0.0           474         203   2000
1.0           408         175   1746
8.0           265         113   1133
9.0            31          13    130
Dropped 'id' and 'attack_cat' columns


## Network build

In [20]:
def build_model2(n_features: int, n_classes: int = 2,
                 hidden: int = 128, detach: bool = True) -> lw.DetectorSequential:
    return lw.DetectorSequential(
        nn.Linear(n_features, hidden), nn.ReLU(),
        lw.DetectorLayer(nn.Linear(hidden, hidden),
                      detector=lw.default_detector(hidden), detach=detach), nn.ReLU(),
        lw.DetectorLayer(nn.Linear(hidden, 64),
                      detector=lw.default_detector(64), detach=detach), nn.ReLU(),
        nn.Linear(64, n_classes),
    )

def build_model(n_features: int, n_classes: int = 2,
                hidden: int = 128, detach: bool = True) -> lw.DetectorSequential:
    return lw.DetectorSequential(
        nn.LayerNorm(n_features),
        lw.DetectorLayer(nn.Linear(n_features, hidden*2),
                      detector=lw.default_detector(hidden*2), detach=detach), nn.ReLU(),
        nn.Linear(hidden*2, hidden), nn.ReLU(),
        lw.DetectorLayer(nn.Linear(hidden, 64),
                      detector=lw.default_detector(64), detach=detach), nn.ReLU(),
        nn.Linear(64, n_classes),
    )

In [21]:
n_attack = int((y_train == 1).sum())
print(f"UNSW-NB15: \n"
        f"\t{len(X_train)} train - {len(X_val)} val - {len(X_test)} test \n"
        f"\t{len(feature_names)} features \n"
        f"\tcategorical features excluded from attack: {[f for f in pp.CATEGORICAL_COLS if f in feature_names]}\n"
        f"\ttraining set class 1 (attack instances) = {n_attack / len(y_train):.1%}\n"
        f"train attack = {la.TRAIN_ATTACK}  |  eval attack = {la.EVAL_ATTACK}\n")

UNSW-NB15: 
	57632 train - 24700 val - 175341 test 
	42 features 
	categorical features excluded from attack: ['proto', 'service', 'state']
	training set class 1 (attack instances) = 55.1%
train attack = pgd  |  eval attack = pgd



In [ ]:
loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train, y_train),
    batch_size=BATCH_SIZE, shuffle=True,
)

# ---- class weights (dataset has more attack instances)------------------
counts = torch.bincount(y_train, minlength=2).float()
class_weights = (len(y_train) / (2 * counts)).to(device)

# ---- model and optimizer -----------------------------------------------
model = build_model(len(feature_names)).to(device)
optimizer = torch.optim.Adam(params=[
    {"params": list(model.backbone_parameters()), "lr": LR},
    {"params": list(model.detector_parameters()), "lr": LR_DET},
])

# ---- training with early stopping --------------------------------------
print("\n== training ==")
best_val_score, best_state, patience_left = -1.0, None, PATIENCE
for epoch in range(EPOCHS):
    stats = lt.train_epoch(model, loader, optimizer,
                        eps=la.EPS, lambda_det=LAMBDA_DET,
                        task_loss_on_adv=TASK_LOSS_ON_ADV,
                        class_weights=class_weights,
                        attack_mask=attack_mask, attack=la.TRAIN_ATTACK,
                        device=device)
    
    # validation against training attack
    # mean between task accuracy on clean data and detector accuracy
    val = le.evaluate(model, X_val, y_val, eps=la.EPS, attack_mask=attack_mask,
                    attack=la.TRAIN_ATTACK, device=device, threshold=THRESHOLD)
    val_task_acc = val["task_clean"]["acc"]
    val_det_bal = 0.5 * (val["det_clean_acc"] + val["det_adv_acc"])
    val_score = 0.5 * (val_task_acc + val_det_bal)

    print(f"epoch {epoch:3d}:\n"
            f"\ttask loss={stats['task_loss']:.4f}\n"
            f"\tdet loss={stats['det_loss']:.4f}\n"
            f"\ttask clean samples acc={stats['task_clean_acc']:.4f}\n"
            f"\ttask adv samples acc={stats['task_adv_acc']:.4f}\n"
            f"\tdet clean samples acc={stats['det_clean_acc']:.4f}\n"
            f"\tdet adv samples acc={stats['det_adv_acc']:.4f}\n"
            f"\t[val] task acc={val_task_acc:.4f}  det bal acc={val_det_bal:.4f}  "
            f"score={val_score:.4f}")

    # early stopping end condition
    if val_score > best_val_score + MIN_DELTA:
        best_val_score = val_score
        best_state = {k: v.detach().cpu().clone()
                        for k, v in model.state_dict().items()}
        patience_left = PATIENCE
    else:
        patience_left -= 1
        if patience_left == 0:
            if val_task_acc > 0.9 and val_det_bal > 0.9:
                print(f"\tearly stopping: no improvement for {PATIENCE} epochs")
                break
            else:
                patience_left = 1

# recover best weights
if best_state is not None:
    model.load_state_dict(best_state)


== training ==
epoch   0:
	task loss=0.2728
	det loss=0.8769
	task clean samples acc=0.8693
	task adv samples acc=0.6431
	det clean samples acc=0.8760
	det adv samples acc=0.8707
	[val] task acc=0.9220  det bal acc=0.9718  score=0.9469


In [ ]:
# ---- detector threshold choice on training attack ----------------------
best_threshold, val_bal = lt.select_threshold(
    model, X_val, y_val, eps=la.EPS, attack_mask=attack_mask,
    attack=la.TRAIN_ATTACK, device=device
)
print(f"\ndetector threshold choice: {best_threshold:.3f} "
        f"(balanced acc on validation set = {val_bal:.4f})")


detector threshold choice: 0.470 (balanced acc on validation set = 0.8272)


In [ ]:
# ---- test set evaluation -----------------------------------------------
print(f"\n== evaluation on test set (eval attack = {la.EVAL_ATTACK}) ==")
m = le.evaluate(model, X_test, y_test, eps=la.EPS, attack_mask=attack_mask,
                attack=la.EVAL_ATTACK, device=device, threshold=best_threshold)

tc, ta, det = m["task_clean"], m["task_adv"], m["detector"]
print("TASK (positive = attack)")
print(f"  clean       : acc={tc['acc']:.4f}  prec={tc['precision']:.4f}  rec={tc['recall']:.4f}")
print(f"  adversarial : acc={ta['acc']:.4f}  prec={ta['precision']:.4f}  rec={ta['recall']:.4f}")
print("DETECTOR (positive = adversarial)")
print(f"  clean accuracy      : {m['det_clean_acc']:.4f}")
print(f"  adversarial accuracy: {m['det_adv_acc']:.4f}")
print(f"  precision / recall  : {det['precision']:.4f} / {det['recall']:.4f}")
print(f"  mean clean/adv score: {m['score_clean']:.4f} / {m['score_adv']:.4f}")

# ---- checkpoint ---------------------------------------------------------
torch.save({"state_dict": model.state_dict(),
            "feature_names": feature_names,
            "attack_mask": attack_mask,
            "threshold": best_threshold}, CHECKPOINT)
print(f"\ncheckpoint saved in {CHECKPOINT}")


== evaluation on test set (eval attack = pgd_adaptive) ==
TASK (positive = attack)
  clean       : acc=0.8828  prec=0.8534  rec=0.9504
  adversarial : acc=0.6872  prec=0.6981  rec=0.7610
DETECTOR (positive = adversarial)
  clean accuracy      : 0.7267
  adversarial accuracy: 0.8991
  precision / recall  : 0.7669 / 0.8991
  mean clean/adv score: 0.3445 / 0.6590

checkpoint saved in modello_detector.pt
